# Calculate mortality at each grid point using central estimates only

$M(x, y) = POP(x, y) \; \times \; BMR_c \; \times \; AF(x, y)  $

This script calculates gridpoint mortality for each mortality health outcome. The final step calculates the sum of the six outcomes to estimate total PM2.5 mortality

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
from utils.mortality_utils import mortality
from utils.utils import get_scenario_config, create_global_country_map

In [ ]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"

In [ ]:
# === Load data ===
bmr_file = "GBD_BMR_Country_COPD_newlabels_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
bmr_country = xr.open_dataarray(bmr_path).sel(quantile="mean")  # central estimate
BMR = create_global_country_map(bmr_country)

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(BMR, method="nearest", tolerance=1e-9)

In [ ]:
# TMREL from GBD 2021
TMREL = 4.15  # central estimate [95% Uniform CI 2.4 – 5.9]

In [ ]:
# === Health variables ===
# COPD, DIABETES, ISCHEMIC_HEART_DISEASE, LOWER_RESPIRATORY_INFECTIONS, LUNG_CANCER, STROKE
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"]

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]
dates = f"{years.start}-{years.stop}"

RR_DIR = "/glade/work/awells/workflow/GBD21/RR_curves/"
PM25_DIR = f"/glade/work/awells/workflow/{model}/pm25/annual_pm25_bc/"
SAVE_DIR = f"/glade/work/awells/workflow/{model}/mortality/pm25/gridpoint_mortality/"

# === Main loop ===
for health_VAR in health_vars:
    
    RR_file = f"IHME_GBD_2021_AIR_POLLUTION_1990_2021_PM_RR_{health_VAR}.nc"
    RR_path = os.path.join(RR_DIR, RR_file)
    RR_values = xr.open_dataset(RR_path)["mean"]  # central estimate

    # Scale the RR to the TMREL so that RR below the TMREL=1
    logRR = np.log(RR_values)
    # Find the log(RR) at the TMREL
    logRR_tmrel = logRR.sel(exposure=TMREL, method="nearest")

    # Shift the function by the log(RR)_TMREL so that log(RR)=0 at TMREL
    logRR_shifted = logRR - logRR_tmrel

    # Set log(RR) below TMREL as 0 and exponentiate to get RR
    scaled_RR = np.exp(logRR_shifted.where(logRR_shifted["exposure"] >= TMREL, 0))
    
    for ens_num in ensemble_members:
        print(f"Processing {scenario}, Ensemble {ens_num:02d}, {health_VAR}")

        pm25_file = f"Annual_PM25_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        pm25_path = os.path.join(PM25_DIR, pm25_file)
        pm25 = xr.open_dataarray(pm25_path)

        # Adjust indices to match (with small tolerance)
        # e.g., max 1e-7 km distance
        pm25 = pm25.reindex_like(BMR, method="nearest", tolerance=1e-9, fill_value=0)

        M = []

        print(f"Processing years {years.start}-{years.stop}")
        for year in range(years.start, years.stop + 1):
            POP = pop.sel(year=year)

            # Find RR at each grid point
            RR = scaled_RR.interp(exposure=pm25.sel(year=year))
            # Calculate the attributable fraction
            AF = 1 - (1/RR)

            del RR

            # Calculate mortality at each grid point
            mortality_year = mortality(AF, BMR, POP)
            M.append(mortality_year)

            del mortality_year, AF, POP

        M_cleaned = [da.drop_vars("year", errors="ignore") for da in M]

        del M

        mortality_timeseries = xr.concat(
            M_cleaned,
            dim=(xr.DataArray(pm25["year"].values,
                              dims="year", name="year"))
        )

        mortality_timeseries = mortality_timeseries.drop_vars(
            ["exposure", "quantile", "region"]
        )

        del M_cleaned

        description = (f"Total {health_VAR} mortality due to PM2.5 using "
                       "central estimates only - scripts "
                       "by A.F. Wells (2025)")
        mortality_timeseries.attrs["description"] = description
        mortality_timeseries.attrs["health_var"] = health_VAR
        mortality_timeseries.attrs["ensemble_number"] = ens_num
        mortality_timeseries.attrs["scenario"] = scenario
        mortality_timeseries.attrs["model"] = model

        out_file = f"Mortality_{health_VAR}_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving mortality timeseries to {out_path}")
        mortality_timeseries.to_netcdf(out_path)

    del RR_values, logRR, logRR_shifted, scaled_RR

print("All processing complete.")

## Save the sum of all mortality outcomes

In [ ]:
for ens_num in ensemble_members:
    # Find all files for this ensemble
    in_files = f"Mortality_*_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    in_path = os.path.join(SAVE_DIR, in_files)
    files = sorted(glob.glob(in_path))

    # Open and combine
    datasets = [xr.open_dataarray(f) for f in files]

    # Align (important in case of slight coordinate mismatches)
    aligned = xr.align(*datasets, join="exact")

    # Sum across the health variables
    summed_da = sum(aligned)

    description = ("Total mortality due to PM2.5 using "
                   "central estimates only - scripts "
                   "by A.F. Wells (2025)")
    summed_da.attrs["description"] = description
    summed_da.attrs["ensemble_number"] = ens_num
    summed_da.attrs["scenario"] = scenario
    summed_da.attrs["model"] = model

    out_file = f"Mortality_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving summed mortality timeseries to {out_path}")
    summed_da.to_netcdf(out_path)

print("All processing complete.")